# 05 - Wrapper Feature Selection

This notebook evaluates probabilistic classifiers after applying a wrapper
feature-selection strategy.

Unlike filter methods, a wrapper evaluates candidate feature subsets using the
performance of a specific classifier.

Therefore, feature selection is performed independently for each classifier.

Sequential Forward Selection is used because it is a wrapper search strategy
covered in the course.

The complete wrapper-selection process is nested inside the outer
cross-validation procedure to avoid optimistic performance estimates.

In [1]:
import pandas as pd
import numpy as np

from scipy.special import logsumexp

from sklearn.base import (
    BaseEstimator,
    ClassifierMixin,
    clone
)

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)

from sklearn.pipeline import Pipeline

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

from sklearn.linear_model import LogisticRegression

from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 1. Dataset Preparation

The same dataset preparation used in the previous PA2 experiments is retained.

The wrapper starts from the 30 valid original predictors.

Feature selection is performed independently for every classifier and every
outer training fold.

In [2]:
DATA_PATH = "../../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(DATA_PATH)

columns_to_remove = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

df_model = df.drop(
    columns=columns_to_remove
)

X = df_model.drop(
    columns=["Attrition"]
)

y = df_model["Attrition"].map({
    "No": 0,
    "Yes": 1
})

print("Predictor matrix shape:", X.shape)
print("Target shape:", y.shape)

Predictor matrix shape: (1470, 30)
Target shape: (1470,)


In [3]:
ordinal_features = [
    "Education",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "JobLevel",
    "JobSatisfaction",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "WorkLifeBalance"
]

numerical_features = (
    X.select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X.select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

continuous_features = [
    feature
    for feature in numerical_features
    if feature not in ordinal_features
]

discrete_features = (
    ordinal_features
    + categorical_features
)

In [4]:
class MixedNaiveBayes(
    BaseEstimator,
    ClassifierMixin
):
    def __init__(
        self,
        continuous_features,
        discrete_features,
        alpha=1.0,
        var_smoothing=1e-9
    ):
        self.continuous_features = continuous_features
        self.discrete_features = discrete_features
        self.alpha = alpha
        self.var_smoothing = var_smoothing

    def fit(self, X, y):

        X = X.copy()
        y = np.asarray(y)

        self.classes_ = np.unique(y)

        selected_columns = (
            X.columns.tolist()
        )

        self.active_continuous_features_ = [
            feature
            for feature in self.continuous_features
            if feature in selected_columns
        ]

        self.active_discrete_features_ = [
            feature
            for feature in self.discrete_features
            if feature in selected_columns
        ]

        self.class_prior_ = {}
        self.gaussian_parameters_ = {}
        self.discrete_probabilities_ = {}
        self.discrete_default_probabilities_ = {}

        n_samples = len(y)

        for cls in self.classes_:

            class_mask = (
                y == cls
            )

            X_cls = X.loc[
                class_mask
            ]

            class_count = (
                class_mask.sum()
            )

            self.class_prior_[cls] = (
                class_count
                / n_samples
            )

            self.gaussian_parameters_[cls] = {}

            for feature in (
                self.active_continuous_features_
            ):

                values = (
                    X_cls[feature]
                    .astype(float)
                    .to_numpy()
                )

                mean = (
                    values.mean()
                )

                variance = (
                    values.var()
                    + self.var_smoothing
                )

                self.gaussian_parameters_[
                    cls
                ][feature] = (
                    mean,
                    variance
                )

            self.discrete_probabilities_[cls] = {}
            self.discrete_default_probabilities_[cls] = {}

            for feature in (
                self.active_discrete_features_
            ):

                all_categories = (
                    X[feature]
                    .astype(str)
                    .unique()
                )

                class_values = (
                    X_cls[feature]
                    .astype(str)
                )

                counts = (
                    class_values
                    .value_counts()
                )

                n_categories = (
                    len(all_categories)
                    + 1
                )

                denominator = (
                    class_count
                    + self.alpha
                    * n_categories
                )

                probabilities = {}

                for category in all_categories:

                    probabilities[
                        category
                    ] = (
                        counts.get(
                            category,
                            0
                        )
                        + self.alpha
                    ) / denominator

                self.discrete_probabilities_[
                    cls
                ][feature] = (
                    probabilities
                )

                self.discrete_default_probabilities_[
                    cls
                ][feature] = (
                    self.alpha
                    / denominator
                )

        return self

    def _joint_log_likelihood(
        self,
        X
    ):

        X = X.copy()

        joint_log_likelihood = []

        for cls in self.classes_:

            log_probability = np.full(
                X.shape[0],
                np.log(
                    self.class_prior_[
                        cls
                    ]
                ),
                dtype=float
            )

            for feature in (
                self.active_continuous_features_
            ):

                mean, variance = (
                    self.gaussian_parameters_[
                        cls
                    ][feature]
                )

                values = (
                    X[feature]
                    .astype(float)
                    .to_numpy()
                )

                log_probability += (
                    -0.5
                    * np.log(
                        2
                        * np.pi
                        * variance
                    )
                    - (
                        (values - mean) ** 2
                        / (2 * variance)
                    )
                )

            for feature in (
                self.active_discrete_features_
            ):

                probabilities = (
                    self.discrete_probabilities_[
                        cls
                    ][feature]
                )

                default_probability = (
                    self.discrete_default_probabilities_[
                        cls
                    ][feature]
                )

                values = (
                    X[feature]
                    .astype(str)
                    .to_numpy()
                )

                feature_probabilities = np.array([
                    probabilities.get(
                        value,
                        default_probability
                    )
                    for value in values
                ])

                log_probability += np.log(
                    feature_probabilities
                )

            joint_log_likelihood.append(
                log_probability
            )

        return np.column_stack(
            joint_log_likelihood
        )

    def predict_proba(
        self,
        X
    ):

        joint_log_likelihood = (
            self._joint_log_likelihood(
                X
            )
        )

        log_normalizer = (
            logsumexp(
                joint_log_likelihood,
                axis=1,
                keepdims=True
            )
        )

        return np.exp(
            joint_log_likelihood
            - log_normalizer
        )

    def predict(
        self,
        X
    ):

        probabilities = (
            self.predict_proba(
                X
            )
        )

        return self.classes_[
            np.argmax(
                probabilities,
                axis=1
            )
        ]

In [5]:
def multiclass_brier_score(
    y_true,
    probabilities,
    classes
):

    y_true = np.asarray(
        y_true
    )

    indicator_matrix = np.zeros_like(
        probabilities,
        dtype=float
    )

    for index, cls in enumerate(
        classes
    ):

        indicator_matrix[:, index] = (
            y_true == cls
        ).astype(float)

    squared_error = (
        probabilities
        - indicator_matrix
    ) ** 2

    return np.mean(
        squared_error.sum(axis=1)
    )

## 2. Classifier-Specific Model Construction

Wrapper selection requires repeatedly fitting a classifier on different
candidate subsets.

A helper function is therefore used to build the appropriate preprocessing and
classifier for each candidate subset.

The same model-specific representations defined in the baseline experiment are
retained.

In [6]:
def build_model(
    model_name,
    selected_features
):

    selected_continuous = [
        feature
        for feature in continuous_features
        if feature in selected_features
    ]

    selected_ordinal = [
        feature
        for feature in ordinal_features
        if feature in selected_features
    ]

    selected_categorical = [
        feature
        for feature in categorical_features
        if feature in selected_features
    ]

    selected_numeric = (
        selected_continuous
        + selected_ordinal
    )

    selected_discrete = (
        selected_ordinal
        + selected_categorical
    )

    if model_name == "Mixed Naive Bayes":

        return MixedNaiveBayes(
            continuous_features=selected_continuous,
            discrete_features=selected_discrete,
            alpha=1.0
        )

    if model_name == "Logistic Regression":

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    StandardScaler(),
                    selected_numeric
                ),
                (
                    "cat",
                    OneHotEncoder(
                        drop="first",
                        handle_unknown="ignore",
                        sparse_output=False
                    ),
                    selected_categorical
                )
            ]
        )

        return Pipeline(
            steps=[
                (
                    "preprocessing",
                    preprocessor
                ),
                (
                    "classifier",
                    LogisticRegression(
                        max_iter=2000,
                        random_state=42
                    )
                )
            ]
        )

    if model_name in ["LDA", "QDA"]:

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    StandardScaler(),
                    selected_numeric
                ),
                (
                    "cat",
                    OrdinalEncoder(
                        handle_unknown="use_encoded_value",
                        unknown_value=-1
                    ),
                    selected_categorical
                )
            ]
        )

        classifier = (
            LinearDiscriminantAnalysis()
            if model_name == "LDA"
            else QuadraticDiscriminantAnalysis()
        )

        return Pipeline(
            steps=[
                (
                    "preprocessing",
                    preprocessor
                ),
                (
                    "classifier",
                    classifier
                )
            ]
        )

    raise ValueError(
        f"Unknown model: {model_name}"
    )

## 3. Wrapper Evaluation Criterion

Sequential Forward Selection evaluates candidate subsets using the F1-measure.

F1 is used because the target is imbalanced and the objective is not simply to
maximize overall accuracy.

Candidate subsets are evaluated through an internal stratified 5-fold
cross-validation performed only within the outer training partition.

In [7]:
def forward_wrapper_selection(
    model_name,
    X_train,
    y_train,
    max_features=15,
    inner_cv=None
):

    if inner_cv is None:

        inner_cv = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=42
        )

    selected = []
    remaining = X_train.columns.tolist()

    best_subset = []
    best_score = -np.inf

    history = []

    while (
        remaining
        and len(selected) < max_features
    ):

        candidate_scores = []

        for feature in remaining:

            candidate_subset = (
                selected
                + [feature]
            )

            model = build_model(
                model_name,
                candidate_subset
            )

            score = cross_val_score(
                model,
                X_train[
                    candidate_subset
                ],
                y_train,
                cv=inner_cv,
                scoring="f1",
                n_jobs=-1
            ).mean()

            candidate_scores.append(
                (
                    feature,
                    score
                )
            )

        best_feature, current_score = max(
            candidate_scores,
            key=lambda item: item[1]
        )

        selected.append(
            best_feature
        )

        remaining.remove(
            best_feature
        )

        history.append({
            "Step": len(selected),
            "AddedFeature": best_feature,
            "InnerF1": current_score
        })

        if current_score > best_score:

            best_score = current_score
            best_subset = selected.copy()

    return (
        best_subset,
        best_score,
        pd.DataFrame(history)
    )

## 4. Nested Cross-Validation Structure

An outer stratified 10-fold cross-validation estimates final performance.

Within every outer training fold, Sequential Forward Selection performs an
independent internal 5-fold cross-validation.

This separates feature-subset search from final validation and avoids using
the outer validation fold during wrapper optimization.

In [8]:
outer_cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [9]:
def evaluate_model(
    model_name,
    selected_features,
    X_train,
    X_test,
    y_train,
    y_test,
    fold
):

    model = build_model(
        model_name,
        selected_features
    )

    model.fit(
        X_train[selected_features],
        y_train
    )

    y_pred = model.predict(
        X_test[selected_features]
    )

    probabilities = model.predict_proba(
        X_test[selected_features]
    )

    classes = model.classes_

    positive_index = np.where(
        classes == 1
    )[0][0]

    positive_probability = (
        probabilities[
            :,
            positive_index
        ]
    )

    return {
        "Model": model_name,
        "Fold": fold,
        "NumberSelected":
            len(selected_features),

        "Accuracy":
            accuracy_score(
                y_test,
                y_pred
            ),

        "Precision":
            precision_score(
                y_test,
                y_pred,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_test,
                y_pred
            ),

        "Specificity":
            recall_score(
                y_test,
                y_pred,
                pos_label=0
            ),

        "F1":
            f1_score(
                y_test,
                y_pred,
                zero_division=0
            ),

        "AUC":
            roc_auc_score(
                y_test,
                positive_probability
            ),

        "Brier":
            multiclass_brier_score(
                y_test,
                probabilities,
                classes
            )
    }

## 5. Wrapper Feature-Selection Experiment

The wrapper procedure is run independently for every classifier and every outer
cross-validation fold.

This means that each classifier can select a different predictor subset.

The selected subsets and internal wrapper scores are stored for later
interpretation.

In [10]:
model_names = [
    "Mixed Naive Bayes",
    "Logistic Regression",
    "LDA",
    "QDA"
]

wrapper_results = []
selection_records = []

for fold, (
    train_index,
    test_index
) in enumerate(
    outer_cv.split(X, y),
    start=1
):

    print(
        f"\nOuter fold {fold}/10"
    )

    X_train = (
        X.iloc[
            train_index
        ].copy()
    )

    X_test = (
        X.iloc[
            test_index
        ].copy()
    )

    y_train = (
        y.iloc[
            train_index
        ].copy()
    )

    y_test = (
        y.iloc[
            test_index
        ].copy()
    )

    for model_name in model_names:

        print(
            f"  Selecting for {model_name}..."
        )

        (
            selected_features,
            inner_score,
            history
        ) = forward_wrapper_selection(
            model_name=model_name,
            X_train=X_train,
            y_train=y_train,
            max_features=15,
            inner_cv=inner_cv
        )

        selection_records.append({
            "Fold": fold,
            "Model": model_name,
            "SelectedFeatures":
                selected_features,
            "NumberSelected":
                len(selected_features),
            "BestInnerF1":
                inner_score
        })

        wrapper_results.append(
            evaluate_model(
                model_name=model_name,
                selected_features=
                    selected_features,
                X_train=X_train,
                X_test=X_test,
                y_train=y_train,
                y_test=y_test,
                fold=fold
            )
        )

print(
    "\nWrapper evaluation completed."
)


Outer fold 1/10
  Selecting for Mixed Naive Bayes...
  Selecting for Logistic Regression...
  Selecting for LDA...
  Selecting for QDA...

Outer fold 2/10
  Selecting for Mixed Naive Bayes...
  Selecting for Logistic Regression...
  Selecting for LDA...
  Selecting for QDA...

Outer fold 3/10
  Selecting for Mixed Naive Bayes...
  Selecting for Logistic Regression...
  Selecting for LDA...
  Selecting for QDA...

Outer fold 4/10
  Selecting for Mixed Naive Bayes...
  Selecting for Logistic Regression...
  Selecting for LDA...
  Selecting for QDA...

Outer fold 5/10
  Selecting for Mixed Naive Bayes...
  Selecting for Logistic Regression...
  Selecting for LDA...
  Selecting for QDA...

Outer fold 6/10
  Selecting for Mixed Naive Bayes...
  Selecting for Logistic Regression...
  Selecting for LDA...
  Selecting for QDA...

Outer fold 7/10
  Selecting for Mixed Naive Bayes...
  Selecting for Logistic Regression...
  Selecting for LDA...
  Selecting for QDA...

Outer fold 8/10
  Selectin

In [11]:
wrapper_fold_results = pd.DataFrame(
    wrapper_results
)

display(
    wrapper_fold_results.round(4)
)

,Model,Fold,NumberSelected,Accuracy,Precision,Recall,Specificity,F1,AUC,Brier
0,Mixed Naive Bayes,1,12,0.8095,0.4286,0.5000,0.8699,0.4615,0.8211,0.2492
1,Logistic Regression,1,15,0.8571,0.7143,0.2083,0.9837,0.3226,0.8476,0.2037
2,LDA,1,15,0.8435,0.5455,0.2500,0.9593,0.3429,0.7327,0.2391
3,QDA,1,9,0.8299,0.4615,0.2500,0.9431,0.3243,0.6369,0.2846
4,Mixed Naive Bayes,2,12,0.7755,0.3200,0.3333,0.8618,0.3265,0.6714,0.3124
5,Logistic Regression,2,13,0.8435,0.5385,0.2917,0.9512,0.3784,0.7873,0.2249
6,LDA,2,15,0.8231,0.4167,0.2083,0.9431,0.2778,0.7971,0.2361
7,QDA,2,15,0.8367,0.5000,0.3333,0.9350,0.4000,0.7690,0.2309
8,Mixed Naive Bayes,3,15,0.8367,0.5000,0.4167,0.9187,0.4545,0.7219,0.2500
9,Logistic Regression,3,15,0.8503,0.5833,0.2917,0.9593,0.3889,0.8150,0.1997


In [12]:
selection_records_df = pd.DataFrame(
    selection_records
)

display(
    selection_records_df
)

,Fold,Model,SelectedFeatures,NumberSelected,BestInnerF1
0,1,Mixed Naive Bayes,"[Age, JobRole, JobLevel, TotalWorkingYears, Ov...",12,0.511865
1,1,Logistic Regression,"[Age, OverTime, MaritalStatus, MonthlyIncome, ...",15,0.480081
2,1,LDA,"[Age, OverTime, MaritalStatus, JobInvolvement,...",15,0.411406
3,1,QDA,"[Age, OverTime, StockOptionLevel, JobLevel, Pe...",9,0.421397
4,2,Mixed Naive Bayes,"[Age, JobRole, JobLevel, OverTime, YearsInCurr...",12,0.540003
5,2,Logistic Regression,"[Age, OverTime, MaritalStatus, JobLevel, Busin...",13,0.495960
6,2,LDA,"[Age, OverTime, JobInvolvement, YearsInCurrent...",15,0.448665
7,2,QDA,"[Age, OverTime, MonthlyIncome, JobLevel, Dista...",15,0.505624
8,3,Mixed Naive Bayes,"[Age, JobRole, JobLevel, OverTime, MonthlyInco...",15,0.531392
9,3,Logistic Regression,"[Age, OverTime, MaritalStatus, YearsInCurrentR...",15,0.512768


## 6. Cross-Validation Summary

The mean and standard deviation of each performance metric are calculated
across the ten outer validation folds.

In [13]:
metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "AUC",
    "Brier"
]

summary_rows = []

for model_name in (
    wrapper_fold_results[
        "Model"
    ].unique()
):

    model_data = (
        wrapper_fold_results[
            wrapper_fold_results[
                "Model"
            ] == model_name
        ]
    )

    row = {
        "Model": model_name,
        "MeanSelectedFeatures":
            model_data[
                "NumberSelected"
            ].mean()
    }

    for metric in metrics:

        row[f"{metric}_mean"] = (
            model_data[
                metric
            ].mean()
        )

        row[f"{metric}_std"] = (
            model_data[
                metric
            ].std()
        )

    summary_rows.append(
        row
    )

wrapper_results_summary = pd.DataFrame(
    summary_rows
)

display(
    wrapper_results_summary.round(4)
)

,Model,MeanSelectedFeatures,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,Specificity_mean,Specificity_std,F1_mean,F1_std,AUC_mean,AUC_std,Brier_mean,Brier_std
0,Mixed Naive Bayes,13.1,0.8293,0.0243,0.4730,0.0771,0.4087,0.0649,0.9099,0.0268,0.4358,0.0610,0.7701,0.0583,0.2400,0.0333
1,Logistic Regression,14.6,0.8714,0.0261,0.7287,0.1747,0.3551,0.0870,0.9708,0.0259,0.4696,0.0999,0.8206,0.0451,0.1976,0.0245
2,LDA,13.5,0.8503,0.0224,0.5810,0.1381,0.2621,0.0816,0.9635,0.0145,0.3589,0.0991,0.7844,0.0370,0.2207,0.0213
3,QDA,12.9,0.8361,0.0093,0.4910,0.0554,0.3293,0.1086,0.9335,0.0249,0.3848,0.0759,0.7165,0.0526,0.2535,0.0221


In [14]:
wrapper_summary = (
    wrapper_results_summary[
        [
            "Model",
            "MeanSelectedFeatures",
            "Accuracy_mean",
            "Precision_mean",
            "Recall_mean",
            "Specificity_mean",
            "F1_mean",
            "AUC_mean",
            "Brier_mean"
        ]
    ].copy()
)

wrapper_summary.columns = [
    "Model",
    "MeanSelectedFeatures",
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "AUC",
    "Brier"
]

wrapper_summary = (
    wrapper_summary
    .sort_values(
        "F1",
        ascending=False
    )
)

display(
    wrapper_summary.round(4)
)

,Model,MeanSelectedFeatures,Accuracy,Precision,Recall,Specificity,F1,AUC,Brier
1,Logistic Regression,14.6,0.8714,0.7287,0.3551,0.9708,0.4696,0.8206,0.1976
0,Mixed Naive Bayes,13.1,0.8293,0.4730,0.4087,0.9099,0.4358,0.7701,0.2400
3,QDA,12.9,0.8361,0.4910,0.3293,0.9335,0.3848,0.7165,0.2535
2,LDA,13.5,0.8503,0.5810,0.2621,0.9635,0.3589,0.7844,0.2207


## 7. Selection Stability

Because wrapper selection is classifier-specific, selection frequencies are
analysed separately for each classifier.

This allows us to identify predictors that are repeatedly useful for a
particular probabilistic model.

In [15]:
stability_rows = []

for model_name in model_names:

    model_records = (
        selection_records_df[
            selection_records_df[
                "Model"
            ] == model_name
        ]
    )

    frequency = {
        feature: 0
        for feature in X.columns
    }

    for selected_features in (
        model_records[
            "SelectedFeatures"
        ]
    ):

        for feature in selected_features:

            frequency[
                feature
            ] += 1

    for feature, count in frequency.items():

        stability_rows.append({
            "Model": model_name,
            "Feature": feature,
            "SelectedFolds": count
        })

wrapper_stability = pd.DataFrame(
    stability_rows
)

display(
    wrapper_stability[
        wrapper_stability[
            "SelectedFolds"
        ] > 0
    ]
    .sort_values(
        [
            "Model",
            "SelectedFolds",
            "Feature"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
)

,Model,Feature,SelectedFolds
60,LDA,Age,10
74,LDA,MaritalStatus,10
77,LDA,NumCompaniesWorked,10
78,LDA,OverTime,10
70,LDA,JobInvolvement,8
...,...,...,...
91,QDA,BusinessTravel,1
92,QDA,DailyRate,1
102,QDA,JobRole,1
109,QDA,PercentSalaryHike,1


## 8. Comparison with the Full-Feature Baseline

Wrapper results are compared with the full-feature baseline.

Positive changes in F1, Recall, and AUC indicate improvement.

Negative changes in Brier score indicate improvement.

In [16]:
baseline_summary = pd.read_csv(
    "../../results/pa2/baseline_summary.csv"
)

comparison = (
    baseline_summary[
        [
            "Model",
            "F1",
            "Recall",
            "AUC",
            "Brier"
        ]
    ]
    .merge(
        wrapper_summary[
            [
                "Model",
                "F1",
                "Recall",
                "AUC",
                "Brier"
            ]
        ],
        on="Model",
        suffixes=(
            "_Baseline",
            "_Wrapper"
        )
    )
)

comparison["F1_change"] = (
    comparison["F1_Wrapper"]
    - comparison["F1_Baseline"]
)

comparison["Recall_change"] = (
    comparison["Recall_Wrapper"]
    - comparison["Recall_Baseline"]
)

comparison["AUC_change"] = (
    comparison["AUC_Wrapper"]
    - comparison["AUC_Baseline"]
)

comparison["Brier_change"] = (
    comparison["Brier_Wrapper"]
    - comparison["Brier_Baseline"]
)

display(
    comparison.round(4)
)

,Model,F1_Baseline,Recall_Baseline,AUC_Baseline,Brier_Baseline,F1_Wrapper,Recall_Wrapper,AUC_Wrapper,Brier_Wrapper,F1_change,Recall_change,AUC_change,Brier_change
0,Logistic Regression,0.5373,0.4259,0.8375,0.1857,0.4696,0.3551,0.8206,0.1976,-0.0677,-0.0708,-0.0169,0.0118
1,Mixed Naive Bayes,0.5083,0.6453,0.7851,0.2902,0.4358,0.4087,0.7701,0.2400,-0.0725,-0.2366,-0.0150,-0.0503
2,LDA,0.4256,0.3168,0.8146,0.2036,0.3589,0.2621,0.7844,0.2207,-0.0668,-0.0547,-0.0302,0.0172
3,QDA,0.3946,0.3755,0.7211,0.2786,0.3848,0.3293,0.7165,0.2535,-0.0098,-0.0462,-0.0047,-0.0251


## 9. Comparison with Filter Strategies

The wrapper results are compared with the previous univariate and multivariate
filter experiments.

This provides the first direct comparison among all feature-selection
strategies required in PA2.

In [17]:
univariate_summary = pd.read_csv(
    "../../results/pa2/univariate_summary.csv"
)

multivariate_summary = pd.read_csv(
    "../../results/pa2/multivariate_summary.csv"
)

strategy_comparison = pd.concat(
    [
        baseline_summary.assign(
            Strategy="All Features"
        ),
        univariate_summary.assign(
            Strategy="Univariate"
        ),
        multivariate_summary.assign(
            Strategy="Multivariate"
        ),
        wrapper_summary.assign(
            Strategy="Wrapper"
        )
    ],
    ignore_index=True
)

strategy_comparison = (
    strategy_comparison[
        [
            "Strategy",
            "Model",
            "F1",
            "Recall",
            "AUC",
            "Brier"
        ]
    ]
)

display(
    strategy_comparison
    .sort_values(
        "F1",
        ascending=False
    )
    .round(4)
)

,Strategy,Model,F1,Recall,AUC,Brier
0,All Features,Logistic Regression,0.5373,0.4259,0.8375,0.1857
1,All Features,Mixed Naive Bayes,0.5083,0.6453,0.7851,0.2902
12,Wrapper,Logistic Regression,0.4696,0.3551,0.8206,0.1976
4,Univariate,Mixed Naive Bayes,0.4600,0.6033,0.7633,0.3190
13,Wrapper,Mixed Naive Bayes,0.4358,0.4087,0.7701,0.2400
2,All Features,LDA,0.4256,0.3168,0.8146,0.2036
5,Univariate,QDA,0.4240,0.3924,0.7302,0.2628
8,Multivariate,Mixed Naive Bayes,0.3962,0.3531,0.7616,0.2386
3,All Features,QDA,0.3946,0.3755,0.7211,0.2786
14,Wrapper,QDA,0.3848,0.3293,0.7165,0.2535


## 10. Export Results

The wrapper performance, classifier-specific selected subsets, and selection
stability are stored for the final PA2 analysis.

In [18]:
wrapper_fold_results.to_csv(
    "../../results/pa2/wrapper_fold_results.csv",
    index=False
)

wrapper_results_summary.to_csv(
    "../../results/pa2/wrapper_results.csv",
    index=False
)

wrapper_summary.to_csv(
    "../../results/pa2/wrapper_summary.csv",
    index=False
)

selection_records_df.to_csv(
    "../../results/pa2/wrapper_selected_subsets.csv",
    index=False
)

wrapper_stability.to_csv(
    "../../results/pa2/wrapper_selection_frequency.csv",
    index=False
)

print(
    "PA2 wrapper results saved successfully."
)

PA2 wrapper results saved successfully.


## 11. Main Observations

Wrapper selection is classifier-specific and retains, on average, between
approximately 13 and 15 predictors depending on the model.

Logistic Regression obtains the strongest wrapper performance, with mean
F1-measure 0.4696, AUC 0.8206, and Brier score 0.1976. However, these results
remain below its full-feature baseline, where F1 reaches 0.5373 and AUC 0.8375.

Mixed Naive Bayes obtains the highest Recall under the wrapper strategy
(0.4087), but this is substantially lower than its baseline Recall of 0.6453.
Its F1 also decreases from 0.5083 to 0.4358. Nevertheless, its Brier score
improves considerably, from 0.2902 to 0.2400.

LDA shows lower F1, Recall, and AUC than in the full-feature baseline, and its
Brier score also becomes worse.

QDA is the classifier least affected by wrapper selection in terms of F1 and
AUC. Its F1 decreases only slightly from 0.3946 to 0.3848 and AUC from 0.7211
to 0.7165, while its Brier score improves from 0.2786 to 0.2535.

The wrapper does not outperform the strongest full-feature configuration for
any classifier in terms of F1 or AUC.

Compared with the filter-based approaches, wrapper Logistic Regression
provides the best reduced-feature F1 score (0.4696), while univariate Mixed
Naive Bayes provides the highest reduced-feature Recall (0.6033).

Overall, the wrapper identifies classifier-specific subsets but does not
improve the best probabilistic baseline. The full-feature Logistic Regression
remains the strongest overall configuration.